In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

from torch_geometric.transforms import LineGraph
from torch_geometric.utils import coalesce
from torch_geometric.data import Data

from ml_surrogates.data_preprocessing.help_functions import *

In [2]:
# Get the absolute path to the project root
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

# Path to the basecase links and stats
basecase_links_path = os.path.join(project_root, 'data', 'links_and_stats', 'pop_1pct_basecase_average_output_links.geojson')

gdf_basecase_links = gpd.read_file(basecase_links_path)
gdf_basecase_links = gdf_basecase_links.set_crs("EPSG:4326", allow_override=True)

### Data Analysis (Paris)

In [3]:
gdf_basecase_links.columns

Index(['link', 'from_node', 'to_node', 'length', 'freespeed', 'capacity',
       'lanes', 'modes', 'vol_car', 'osm:relation:route_master',
       'osm:way:vehicle', 'osm:way:traffic_calming', 'osm:way:junction',
       'osm:way:motorcycle', 'isUrban', 'osm:way:lanes', 'osm:way:service',
       'osm:way:psv', 'osm:way:id', 'osm:way:access', 'osm:way:oneway',
       'highway', 'osm:relation:route', 'osm:way:railway', 'osm:way:name',
       'storageCapacityUsedInQsim', 'osm:way:tunnel', 'variance', 'cv_percent',
       'std_dev', 'std_dev_multiplied', 'geometry'],
      dtype='object')

In [18]:
gdf_basecase_links.head()

,link,from_node,to_node,length,freespeed,capacity,lanes,modes,vol_car,osm:relation:route_master,...,osm:relation:route,osm:way:railway,osm:way:name,storageCapacityUsedInQsim,osm:way:tunnel,variance,cv_percent,std_dev,std_dev_multiplied,geometry
0,100315,24972409,24972408,16.181257,8.333333,480.0,1.0,"bus,car,car_passenger",51.448980,None,...,bicycle,None,Carrefour de l'Odéon,NaN,None,21.124948,8.933488,4.596188,3.667228,"LINESTRING (2.33869 48.85181, 2.33847 48.85181)"
1,100316,5904976363,24983651,14.860209,8.333333,480.0,1.0,"bus,car,car_passenger,pt",52.979592,None,...,bus,None,Carrefour de l'Odéon,NaN,None,24.897543,9.418237,4.989744,3.981240,"LINESTRING (2.33874 48.85242, 2.33872 48.85229)"
2,100317,24983651,5904976363,14.860209,8.333333,960.0,2.0,"bus,car,car_passenger,pt",23.744898,None,...,bus,None,Carrefour de l'Odéon,NaN,None,5.863494,10.197834,2.421465,1.932050,"LINESTRING (2.33872 48.85229, 2.33874 48.85242)"
3,100321,664205947,24972376,22.264540,8.333333,960.0,2.0,"car,car_passenger",60.071429,None,...,None,None,Boulevard Saint-Germain,NaN,None,28.413265,8.873452,5.330409,4.253051,"LINESTRING (2.33994 48.85200, 2.33986 48.85181)"
4,100324,24972376,24972375,64.853276,8.333333,480.0,1.0,"bus,car,car_passenger",66.020408,None,...,bicycle,None,Rue Dupuytren,NaN,None,30.264890,8.332807,5.501353,4.389445,"LINESTRING (2.33986 48.85181, 2.33909 48.85152)"


In [22]:
gdf_basecase_links['osm:way:name'].value_counts()

osm:way:name
Boulevard Périphérique Intérieur          178
Boulevard Périphérique Extérieur          173
Avenue Daumesnil                          121
Rue de Vaugirard                          101
Ligne de Paris à Saint-Germain-en-Laye     99
                                         ... 
Passage Gambetta                            1
Rue des Rondonneaux                         1
Avenue de Salonique                         1
Rue Nanteuil                                1
Rue du Chaffault                            1
Name: count, Length: 4658, dtype: int64

In [20]:
gdf_basecase_links['osm:way:oneway'].value_counts()

osm:way:oneway
yes    17845
no      1166
-1        13
Name: count, dtype: int64

In [67]:
def form_chain(edges):
    from collections import defaultdict

    # Build forward and reverse maps
    forward_map = defaultdict(list)
    reverse_map = defaultdict(list)
    for u, v in edges:
        
        if u == v:
            continue  # Skip self-loops

        forward_map[u].append(v)
        reverse_map[v].append(u)

    # Find potential start nodes (sources not appearing as targets)
    start_candidates = set(forward_map.keys()) - set(reverse_map.keys())

    print(start_candidates)
    
    # Fallback: arbitrarily choose one if no clear start exists
    if not start_candidates:
        print("⚠️  No clear start node found. Possible cycle. Picking arbitrary start.")
        start = list(forward_map.keys())[0]
    else:
        start = start_candidates.pop()

    # Traverse chain safely
    visited = set()
    chain = [start]
    
    while start in forward_map:

        for next_node in forward_map[start]:
        
            if next_node in visited:
                continue
            
            chain.append(next_node)
            visited.add(start)
            start = next_node

    return chain

In [68]:
gdf_road = gdf_basecase_links[gdf_basecase_links['osm:way:name'] == 'Avenue Daumesnil']
edges = list(zip(gdf_road['from_idx'].astype(int).values, gdf_road['to_idx'].astype(int).values))

chain = form_chain(edges)
print(chain)

{12617, 14378, 14374, 12055}
[12617, 6617, 8484, 6871, 1514, 5956, 6519, 8490, 15768, 15767, 5062, 15766, 6714, 15193, 6430, 1318, 14377]


### Algo Analysis (Dual Graph Transformation)

In [46]:
use_linegraph = True

_, stacked_edge_geometries_tensor, edges_base, nodes = get_link_geometries(gdf_basecase_links)
edge_index = torch.tensor(edges_base, dtype=torch.long).t().contiguous()

linegraph_transformation = LineGraph()

data = Data(edge_index=edge_index)
data.num_nodes = edge_index.shape[1] if use_linegraph else len(nodes)
if use_linegraph:
    data = linegraph_transformation(data)

# data.x = edge_tensor
# data.pos = stacked_edge_geometries_tensor
# data.y = compute_target_tensor_only_edge_features(vol_base_case, gdf)

In [ ]:
# edges = list(zip(edge_index[0].numpy(), edge_index[1].numpy()))
edges = list(zip(data.edge_index[0].numpy(), data.edge_index[1].numpy()))

self_loops = 0
one_way = 0
bidirectional = 0

for (u,v) in tqdm(edges):
    if u == v:
        self_loops += 1
    elif (v, u) in edges:
        bidirectional += 1
    else:
        one_way += 1

print(f"Self-loops: {self_loops}, One-way edges: {one_way}, Bidirectional edges: {bidirectional}")

100%|██████████| 59851/59851 [02:35<00:00, 383.98it/s]

Self-loops: 766, One-way edges: 47293, Bidirectional edges: 11792


In [ ]:
edge_index = coalesce(edge_index)
edges = list(zip(edge_index[0].numpy(), edge_index[1].numpy()))

manual_lineG = list()

for i in tqdm(range(len(edges))):

    (u,v) = edges[i]

    for j in range(len(edges)):

        (x,y) = edges[j]

        if v == x:
            manual_lineG.append((i, j))

100%|██████████| 31559/31559 [03:44<00:00, 140.40it/s]


In [49]:
len(manual_lineG)

59851